# Lesson 48 · 🎨 Time–frequency plots compared — seeing the time–frequency world at a glance: linear vs mel vs log-mel spectrograms side by side

**Goal**: put the linear, mel and log-mel spectrograms side by side and see clearly what actually happens before and after compression.

> **🎨**: fewer formulas, more looking at differences; read it alongside L43–L47.

🔗 Aurora connection: `aurora.audio.stft` (`stft`, `magnitude_spectrogram`) · `aurora.audio.mel` (`mel_filterbank`, `mel_spectrogram`)

← **Previous lesson** · [L47 · Writing the mel filterbank](L47_mel_implement.ipynb)

> Last lesson covered **writing the mel filterbank**: building the log-mel pipeline yourself — STFT → power spectrum → mel filtering → log compression — and matching the repo's output.
> This lesson explores **time–frequency plots compared**.

At heart, the STFT cuts a continuous signal into short frames, runs a DFT on each frame, and lays those spectra out side by side in time order to form a heatmap. The mel filterbank then does a perceptual compression along the frequency axis: low frequencies are kept finer, high frequencies are merged more coarsely, so that the limited number of dimensions goes mostly to the regions the ear is truly sensitive to. Chain these two steps together and you have the most common front-end feature pipeline in audio AI.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import chirp
from aurora.audio.stft import stft, magnitude_spectrogram
from aurora.audio.mel import mel_spectrogram, mel_filterbank

## 1. Framing: a sliding window moving along the signal

### Why frame at all? Spectral leakage and the window function

Intuition: say we have a 10-second song and want to know its frequency content at the 1-second mark. By default the DFT treats the whole 10 seconds as one period. If the signal's edges aren't "natural period boundaries" (for example, 440 Hz doesn't fit a whole number of complete cycles into the 10 seconds), the DFT "leaks" energy into neighboring frequencies in the frequency domain. This is **spectral leakage** — one sharp, single frequency turns into a blurry mix of many.

The beauty of framing: cut the 10-second signal into short, overlapping frames (say 50 ms each) and run a DFT on each one separately. That way each little frame is more likely to hold "a whole number of complete cycles," and leakage goes down. Add a **window function** — letting the signal fade smoothly to 0 at both ends of the frame instead of being chopped off abruptly — and leakage is suppressed even further. Picture a drum hit, "bang": if you cut it out with a rectangular window that starts and stops suddenly, the sharp jumps at the cut points produce extra high-frequency artifacts; with a Hann window fading both ends out smoothly, there's no such jump.

### Framing parameters and formulas

For a signal with sample rate `sr`, define:
- **frame length** `n_fft`: how many samples each frame holds
- **hop length** `hop_length`: the gap between the start points of neighboring frames

**The most basic framing** (`center=False`):
- frame `t` starts at sample index `t * hop_length`
- it covers the interval `[t*hop_length, t*hop_length + n_fft)`
- number of frames: `n_frames = 1 + (N - n_fft) // hop_length`

Deriving the formula:
- frame 0 starts at 0, frame 1 at `hop_length`, ..., frame `t` at `t*hop_length`
- the last sample of the last frame can't go past `N-1`, i.e. `t*hop_length + n_fft - 1 <= N - 1`
- rearranging gives `t*hop_length <= N - n_fft`, so `t <= (N - n_fft) / hop_length`
- the largest `t` is `floor((N - n_fft) / hop_length)`, and the number of frames = `floor(...) + 1 = 1 + (N - n_fft) // hop_length`

**Overlap ratio**: `overlap_ratio = 1 - hop_length / n_fft`. With the typical setting `n_fft=1024, hop_length=256` the overlap is 75% (each frame contributes a quarter of new information; three quarters overlap with the previous frame).

### Window functions: smoothing the edges of a frame

The **Hann window** (a raised-cosine window, a close cousin of the Hamming window) uses the formula `w[n] = 0.5 * (1 - cos(2π n / N))`, where `n = 0, 1, ..., N-1`.

Where does this formula come from? The periodicity of the cosine lets us design a symmetric rise-and-fall shape. At `n=0`, `cos(0)=1`, so `w[0] = 0.5*(1-1) = 0` (it starts at 0); at `n=N/2`, `cos(π)=-1`, so `w[N/2] = 0.5*(1-(-1)) = 1` (it peaks at 1 in the middle); at `n=N-1` (for the DFT-even periodic window), `w[N-1]` is also close to 0 (it falls back at the end). The factor 0.5 makes the peak exactly 1, so the window doesn't attenuate the signal too much.

> **Note**: `aurora.audio.stft` calls `get_window(..., periodic=True)`, with denominator `N` (the frame length itself).
> This is the standard form of the **periodic (DFT-even) window** — it assumes the signal repeats after N samples, so samples 0 and N are the same point in the period.
> Change the denominator to `N-1` and you get the **symmetric window**, which is exactly 0 at both ends. The two look almost identical, but mathematically the periodic window fits the DFT's periodicity assumption better.

### center=True: reflection padding and the change in frame count

**Why center?** With basic framing, frame 0 is `[0:n_fft]`, and the last frame often can't fully cover the samples at the end of the signal. Also, the very first samples of the signal are covered only once, by frame 0, with no other frames to "average" them, so the start of the signal gets uneven weighting.

**The fix**: reflection-pad `n_fft // 2` samples at each end of the signal (that is, "mirror-flip" a segment of length `n_fft//2` and stick it on the front and back), then frame it. That way:
- sample 0 of the original signal now sits at the center of a frame (instead of at an edge)
- the last sample also gets full window coverage

**Why reflect instead of padding with zeros?** Zero padding introduces an artificial "jump" at the signal's edges, which is hard to remove completely even with a window. Reflection (mirroring) keeps the signal continuous: the values don't suddenly jump, so the window's smooth taper works better.

**The change in frame count**: after padding, the signal length becomes `N + 2*(n_fft//2)`. Plug it into the basic frame-count formula:
```
n_frames = 1 + (N_padded - n_fft) // hop_length
         = 1 + (N + 2*(n_fft//2) - n_fft) // hop_length
```
Simplify (assuming `n_fft` is even):
```
N + 2*(n_fft//2) - n_fft = N + n_fft - n_fft = N
```
So:
```
n_frames = 1 + N // hop_length
```
(Note that `N` here is the **original signal length**, not the padded length.) That's why some tools (such as librosa) say that with `center=True` the number of frames is about `1 + signal_length // hop_length`.

> **Note**: the simplification above holds exactly only when `n_fft` is even. If `n_fft` is odd, `2*(n_fft//2) ≠ n_fft` (it's one short), so the formula is slightly different. In practice `n_fft` is always a power of 2 (512, 1024, etc.), so this isn't a problem.

In [ ]:
# Framing demo (a static picture of several overlapping frames)
sr = 8000
duration = 0.05  # 50 ms, easy to visualize
t = np.arange(int(sr * duration)) / sr
signal = np.sin(2 * np.pi * 440 * t) + 0.3 * np.sin(2 * np.pi * 1200 * t)

n_fft = 256
hop_length = 64
n_frames_show = 4

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(t, signal, color='0.6', lw=1, label='Original signal')

colors = plt.cm.tab10(np.linspace(0, 0.4, n_frames_show))
for i in range(n_frames_show):
    start = i * hop_length
    end = start + n_fft
    if end > len(signal):
        break
    hann = 0.5 * (1 - np.cos(2 * np.pi * np.arange(n_fft) / n_fft))  # periodic (DFT-even), same as aurora.audio.stft
    frame_t = t[start:end]
    ax.fill_between(frame_t, signal[start:end] * hann,
                    alpha=0.35, color=colors[i], label=f'Frame {i}')
    ax.axvline(frame_t[0], color=colors[i], lw=0.8, ls='--')

ax.set_xlabel('Time (s)')
ax.set_title(f'Framing  n_fft={n_fft}  hop_length={hop_length}  overlap={1-hop_length/n_fft:.0%}')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

### Demo: spectral leakage vs suppression by a window

Why does a window function suppress leakage? A simple example below shows it at a glance.

In [ ]:
# Demo: spectral leakage and what a window does
# Build a pure 1000 Hz sine, but make the signal length **not** a whole number of 1000 Hz cycles
sr = 8000
duration = 0.1  # 100 ms
t = np.arange(int(sr * duration)) / sr
freq_true = 1000  # true frequency
# At an 8000 Hz sample rate, each cycle of 1000 Hz takes 8 samples
# 100 ms holds about 10 cycles, but we deliberately make it not a whole number
signal_leak = np.sin(2 * np.pi * freq_true * t)

# Compute the FFT (no window)
from numpy.fft import rfft, rfftfreq
n_fft = 512
# One FFT with a rectangular window (no window) and one with a Hann window
rect_spec = np.abs(rfft(signal_leak, n=n_fft))
hann_win = 0.5 * (1 - np.cos(2 * np.pi * np.arange(len(signal_leak)) / len(signal_leak)))
hann_spec = np.abs(rfft(signal_leak * hann_win, n=n_fft))

freqs = rfftfreq(n_fft, 1/sr)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))

# Subplot 1: the original signal
ax = axes[0, 0]
ax.plot(t[:int(0.01*sr)], signal_leak[:int(0.01*sr)], 'b-', lw=1.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title(f'Original signal (pure 1000 Hz sine, {duration*1000:.0f} ms)\nlength is not a whole number of cycles → jumps at the edges')
ax.grid(True, alpha=0.3)

# Subplot 2: the signal times the Hann window
ax = axes[0, 1]
ax.plot(t, signal_leak, 'b-', lw=1, alpha=0.6, label='Original signal')
ax.plot(t, hann_win, 'r-', lw=2, label='Hann window')
ax.plot(t, signal_leak * hann_win, 'g-', lw=1.5, label='Product')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('The window smooths the edges')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

# Subplot 3: spectra compared (dB scale)
ax = axes[1, 0]
ax.plot(freqs, 20*np.log10(rect_spec + 1e-10), 'b-', lw=1.5, label='No window (rectangular)')
ax.plot(freqs, 20*np.log10(hann_spec + 1e-10), 'r-', lw=1.5, label='Hann window')
ax.set_xlim(0, 3000)
ax.set_ylim(-80, 10)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title('Spectral leakage compared (dB)\nno window: clear side lobes; Hann window suppresses them')
ax.axvline(freq_true, color='gray', ls='--', lw=1, alpha=0.7, label='True frequency')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3, which='both')

# Subplot 4: spectra compared (linear scale, zoomed in)
ax = axes[1, 1]
ax.plot(freqs, rect_spec, 'b-', lw=1.5, label='No window (rectangular)')
ax.plot(freqs, hann_spec, 'r-', lw=1.5, label='Hann window')
ax.set_xlim(800, 1200)
ax.set_ylim(0, 300)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude')
ax.set_title('Zoom on the peak\nthe Hann window has a wider main lobe but lower side lobes')
ax.axvline(freq_true, color='gray', ls='--', lw=1, alpha=0.7)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"[What the leakage shows]")
print(f"  • Rectangular window (no window): one main lobe at 1000 Hz, but clear 'side lobes' all around it")
print(f"    — because the jumps at the signal's edges (from nothing to something, and back to nothing) create high-frequency components")
print(f"  • Hann window: multiplies the signal by a smooth function, removing the edge jumps")
print(f"    — the main lobe gets wider (frequency resolution drops a little), but the side lobes are strongly suppressed → less leakage")
print(f"\nIn audio analysis this trade is well worth it: we'd rather give up a little frequency resolution in exchange for less spectral interference.")

## 2b. The mel filterbank: design and standards

Once the mel scale is defined, how do we use it to design filters?

### Building the filterbank

Spread $n_\text{mels}$ center frequencies **evenly along the mel axis** (say 0, 100, 200, ..., up to the top of the mel axis), then **convert them back to the Hz axis**. On the Hz axis, the low-frequency filters end up dense and the high-frequency ones sparse — exactly matching human perception.

Each filter is a **triangle** spanning the range between its two neighboring center frequencies:
$$H_m(k) = \begin{cases}
0 & \text{if } k \leq f_{\text{left}} \text{ or } k \geq f_{\text{right}} \\
\frac{k - f_{\text{left}}}{f_{\text{center}} - f_{\text{left}}} & \text{if } f_{\text{left}} < k \leq f_{\text{center}} \\
\frac{f_{\text{right}} - k}{f_{\text{right}} - f_{\text{center}}} & \text{if } f_{\text{center}} < k < f_{\text{right}}
\end{cases}$$

where $f_{\text{left}}, f_{\text{center}}, f_{\text{right}}$ are three neighboring center frequencies.

**Why triangles?** Simple, fast to compute, and physically intuitive (like the slopes of an analog filter).

### Why is the number of mel channels so often 80?

In audio AI, `n_mels=80` has become the **de facto industry standard**. Why 80?

1. **Rule of thumb**: across the audible range of 20 Hz – 20 kHz, humans can tell apart about 70–150 "pitch levels" (a bit like the 88 keys of a piano). 80 covers this range reasonably.
2. **Matching the sample rate**: at a 16 kHz sample rate (common in speech processing), 80 mel filters cover 0–8000 Hz (the Nyquist frequency); at 22.05 kHz (music), they cover 0–11 kHz. This spread captures most of what the ear cares about.
3. **Model conventions**: once papers, datasets and pretrained models all use 80, the whole ecosystem leans toward that standard.

> **Note**: 80 isn't "optimal." Depending on the task (music vs speech), the sample rate and the compute budget, you can go to 40, 128 and so on. But if you want to reuse open-source model weights, 80 is the safest choice.

### HTK convention vs Slaney normalization

The normalization choice affects **how comparable the energies of different mel filters are**.

**HTK convention** (Aurora's default, used by many speech AI systems):
- every triangle's **peak is set to 1.0** (i.e. constant height)
- high-frequency triangles are wider (because the spacing on the Hz axis is larger), so they **cover more area** and collect more energy
- this is **by design** — the aim is to emphasize the absolute value of high-frequency energy rather than a normalized relative strength

**Slaney normalization**:
- each triangle is scaled by 2 / its **bandwidth** (`right_hz - left_hz`), so that every area is 1
- result: the output energies of different mel filters are more directly comparable
- used in some music applications, so the high frequencies don't get "drowned out"

| Aspect | HTK | Slaney |
|------|-----|--------|
| Peak | 1.0 | 2.0 / bandwidth |
| Area | small at low frequencies, large at high | all equal |
| When to use | speech recognition, general | music analysis |

> **Practical advice**: start with Aurora's default (HTK) unless you're reproducing a paper that uses Slaney.

## 2a. Where the mel scale comes from: the ear's nonlinear perception

### What is a "mel"? Where does the formula come from?

A very fair question! The **mel** isn't derived from a formula — it's distilled from **psychoacoustic experiments**.

Human hearing isn't linear. Imagine playing a 100 Hz bass note at home: down there, going to roughly 200 Hz really does sound about "twice as high" — the ear is close to linear at low frequencies. But start from 1000 Hz, and to sound "twice as high" you have to climb to about 3500 Hz. That shows our ears "compress" the high range — differences between high notes are less noticeable than between low ones.

The **mel scale** is a **psychological pitch scale**, defined as:
$$\text{mel} = 2595 \cdot \log_{10}\left(1 + \frac{f}{700}\right)$$

where $f$ is the frequency in Hz. Where does this formula come from?

1. **History**: in 1937, Stevens and Volkmann ran hearing experiments in which listeners adjusted a frequency until the pitch intervals **sounded equal**.
2. **Fitting**: the experimental data were fitted with a logarithmic function, and this formula is the result of that (later) fit.
3. **What the parameters mean**:
   - `2595` is a normalization factor that makes 1000 Hz correspond to exactly 1000 mel (easy to remember)
   - `700` Hz is the "corner frequency," roughly where the ear's perception changes character (below 700 Hz perception is fairly linear; above 700 Hz it becomes clearly nonlinear)

**The intuition behind the log form**: a logarithm naturally "compresses" high frequencies — the gap between $\log(10000)$ and $\log(5000)$ is far smaller than the gap between $10000$ and $5000$. That matches the ear's falling resolution at high frequencies.

### Checking the properties

Let's look at how this formula actually behaves.

In [ ]:
# Check the properties of the mel formula
def hz_to_mel(f):
    return 2595 * np.log10(1 + f / 700)

def mel_to_hz(m):
    return 700 * (10 ** (m / 2595) - 1)

# Try a few key frequencies
test_freqs = np.array([0, 100, 700, 1000, 2000, 4000, 8000, 16000])
test_mels = hz_to_mel(test_freqs)

print("[Mel formula check]")
print("Frequency (Hz)  →  mel")
print("-" * 30)
for f, m in zip(test_freqs, test_mels):
    print(f"{f:6.0f}  →  {m:8.1f}")

print("\n[Key observations]")
print(f"  • 100 Hz → {hz_to_mel(100):.1f} mel")
print(f"  • 700 Hz → {hz_to_mel(700):.1f} mel (corner frequency)")
print(f"  • 1000 Hz → {hz_to_mel(1000):.1f} mel (reference point, exactly 1000)")
print(f"\n  • 200 Hz (2× 100) → {hz_to_mel(200):.1f} mel")
print(f"  • 2000 Hz (2× 1000) → {hz_to_mel(2000):.1f} mel")
print(f"  • 4000 Hz (2× 2000) → {hz_to_mel(4000):.1f} mel")
print(f"\n  → low frequencies: mel gain from one doubling ≈ {hz_to_mel(200) - hz_to_mel(100):.0f}")
print(f"  → high frequencies: mel gain from one doubling ≈ {hz_to_mel(4000) - hz_to_mel(2000):.0f}")
print(f"\nSo mel is not a pure log scale: close to linear at low frequencies (a doubling adds few mels), close to logarithmic at high ones (each doubling adds a similar, larger amount)")

# Plot: linear frequency vs mel frequency
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

freqs_plot = np.linspace(0, 8000, 1000)
mels_plot = hz_to_mel(freqs_plot)

# Left: linear scale
ax = axes[0]
ax.plot(freqs_plot, mels_plot, 'b-', lw=2)
ax.scatter(test_freqs[test_freqs <= 8000], test_mels[test_freqs <= 8000], color='red', s=50, zorder=5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Mel')
ax.set_title('Hz-to-mel conversion curve\n(log shape: steep at low frequencies, flat at high)')
ax.grid(True, alpha=0.3)

# Right: interval sizes
ax = axes[1]
freq_intervals = [100, 200, 500, 1000, 2000, 4000, 8000]
mel_diffs = []
for i in range(len(freq_intervals) - 1):
    f1, f2 = freq_intervals[i], freq_intervals[i+1]
    m1, m2 = hz_to_mel(f1), hz_to_mel(f2)
    mel_diffs.append(((f1+f2)/2, m2 - m1))

freq_centers = [(freq_intervals[i] + freq_intervals[i+1])/2 for i in range(len(freq_intervals)-1)]
mel_diffs_vals = [hz_to_mel(freq_intervals[i+1]) - hz_to_mel(freq_intervals[i]) for i in range(len(freq_intervals)-1)]

ax.bar(range(len(freq_centers)), mel_diffs_vals, color='steelblue', alpha=0.7)
ax.set_xticks(range(len(freq_centers)))
ax.set_xticklabels([f"{fc:.0f}" for fc in freq_centers], rotation=45, fontsize=9)
ax.set_xlabel('Center of frequency range (Hz)')
ax.set_ylabel('Mel difference across the range')
ax.set_title('Mel difference for each frequency range\n(ranges are not equal width in Hz)')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Compare: filter layout on a linear frequency axis vs a mel axis
sr = 22050
n_fft = 1024
n_mels = 40
fmax = sr // 2

fb = mel_filterbank(n_mels=n_mels, n_fft=n_fft, sample_rate=sr, fmin=0.0, fmax=fmax)
# fb shape: (n_mels, n_fft//2 + 1)
freqs = np.linspace(0, fmax, n_fft // 2 + 1)

def hz_to_mel(f):
    return 2595 * np.log10(1 + f / 700)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: linear frequency axis
for m in range(n_mels):
    axes[0].plot(freqs, fb[m], lw=0.9)
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Weight')
axes[0].set_title(f'Linear frequency axis  n_mels={n_mels}\n(dense at low frequencies, sparse at high)')

# Right: mel frequency axis
mel_freqs = hz_to_mel(freqs)
for m in range(n_mels):
    axes[1].plot(mel_freqs, fb[m], lw=0.9)
axes[1].set_xlabel('Frequency (mel)')
axes[1].set_ylabel('Weight')
axes[1].set_title(f'Mel frequency axis  n_mels={n_mels}\n(evenly spaced)')

plt.tight_layout()
plt.show()

### ⚠️ An important clarification: the "even spacing" on the mel axis comes from changing coordinates, not from changing the filters

Beginners often ask: **"When I switch the x-axis to mel frequency, why do the same filters suddenly look evenly spaced? The data didn't change!"**

The answer: **it's the axis itself that changed, not the filter data.**

An analogy: you have 100 people with heights from 150 cm to 190 cm, spread evenly along a height axis. Now you switch to "the log of height" as the x-axis. On the log axis these 100 people will seem to "bunch up" in places — because the log function "compresses" the gaps between larger numbers. But nobody's height changed; only the coordinate system you view the data in did.

In the same way, the mel filters are unevenly spread on the Hz axis — "dense at low frequencies, sparse at high" — but when you view them on the mel axis (a transformation of log frequency), the filter centers become **evenly spaced**. That's a property of the coordinate system you project onto, not a change in the filters' shapes.

The comparison plot below makes this clear.

## 3. Parameter experiment: linear spectrogram vs mel spectrogram of a chirp

**Signal**: `chirp(sr=22050, duration=2.0, f0=200, f1=8000)` — a linear upward frequency sweep.

**Key parameters**:
- `n_fft=1024`, `hop_length=256`: control the time–frequency resolution trade-off
- `n_mels=80`: the number of mel channels; more channels, more high-frequency detail
- `fmax=8000`: the upper cutoff; energy above this frequency is discarded

**What to expect**:
- Linear spectrogram: the rising sweep is a straight line and **takes up more rows** in the high-frequency region, while low-frequency detail is squeezed
- Mel spectrogram: the rising sweep **climbs steeply at first and then levels off** (low frequencies get more rows), so low-frequency information is richer
- the log-mel dynamic range is about 40–60 dB, and the noise floor is clearer

In [ ]:
sr = 22050
n_fft = 1024
hop_length = 256
n_mels = 80
fmax = 8000

sig = chirp(sample_rate=sr, duration=2.0, f0=200, f1=fmax)

# Linear spectrogram
S = magnitude_spectrogram(sig, n_fft=n_fft, hop_length=hop_length)
# aurora's stft outputs (n_frames, n_freqs); transpose to (n_freqs, n_frames) for plotting
# S is a magnitude (amplitude) spectrum → 20·log10; for a power spectrum use 10·log10
# Add 1e-8 to avoid log(0) — the usual trick for numerical stability
S_db = (20 * np.log10(S + 1e-8)).T

# Mel spectrogram (log scale)
M = mel_spectrogram(sig, sample_rate=sr, n_fft=n_fft, hop_length=hop_length,
                    n_mels=n_mels, fmin=0.0, fmax=fmax)
# M is a power spectrum → 10·log10; 10·log10(|X|²) = 20·log10(|X|), so the values have the same units as S_db
M_db = (10 * np.log10(M + 1e-8)).T  # (n_frames, n_mels) → (n_mels, n_frames) for pcolormesh

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

times = np.arange(S.shape[0]) * hop_length / sr  # S.shape[0]=n_frames
freqs_lin = np.linspace(0, sr / 2, S.shape[1])       # S.shape[1]=n_freqs

im0 = axes[0].pcolormesh(times, freqs_lin / 1000, S_db,
                         shading='auto', cmap='magma',
                         vmin=S_db.max() - 60)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Frequency (kHz)')
axes[0].set_title('Linear magnitude spectrogram (dB)\n✗ the chirp sweep takes up many high-frequency rows')
plt.colorbar(im0, ax=axes[0], label='dB')

times_mel = np.arange(M.shape[0]) * hop_length / sr  # M.shape[0]=n_frames
im1 = axes[1].pcolormesh(times_mel, np.arange(n_mels), M_db,
                         shading='auto', cmap='magma',
                         vmin=M_db.max() - 60)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Mel channel index')
axes[1].set_title(f'Log-mel spectrogram  n_mels={n_mels}\n✓ the chirp sweep spends more channels on low frequencies')
plt.colorbar(im1, ax=axes[1], label='dB')

plt.tight_layout()
plt.show()

print(f'aurora linear spectrogram shape: {S.shape}  (n_frames, n_fft//2+1)')
print(f'aurora mel spectrogram    shape: {M.shape}  (n_frames, n_mels)')

# ────────────────────────────────────────────────────────────────
# [Explaining what we see]
# ────────────────────────────────────────────────────────────────
print("\n[Why does the chirp's rising sweep look different in the two plots?]\n")

# Compute the chirp's frequency at a few moments, and the matching mel values
t_samples = np.array([0, 0.5, 1.0, 1.5, 2.0])  # seconds
f_linear = 200 + (fmax - 200) * t_samples / 2.0  # linear sweep: f(t) = 200 + 3900*t
f_mel = hz_to_mel(f_linear)  # convert to the mel domain

print(f"time (s)  →  linear frequency (Hz)  →  mel")
print("-" * 50)
for t, f_lin, f_mel_val in zip(t_samples, f_linear, f_mel):
    # Work out the matching mel channel index (assuming 0 to n_mels-1 are spread evenly on the mel axis)
    mel_idx = (f_mel_val / hz_to_mel(fmax)) * (n_mels - 1)
    print(f"{t:6.1f}  →  {f_lin:8.0f}  →  mel={f_mel_val:7.1f}  (channel idx≈{mel_idx:.1f})")

print(f"\n[Key observations]")
print(f"  each step in linear frequency: {f_linear[1] - f_linear[0]:.0f} Hz (fixed)")
print(f"  growth in mel: {f_mel[1] - f_mel[0]:.1f}, {f_mel[2] - f_mel[1]:.1f}, {f_mel[3] - f_mel[2]:.1f}, {f_mel[4] - f_mel[3]:.1f} ...")
print(f"\n  → low to mid frequencies: for every {(fmax-200)/2*2/4:.0f} Hz added, mel grows by ~{(f_mel[1] - f_mel[0])*4/2:.0f}")
print(f"  → mid to high frequencies: for every {(fmax-200)/2*2/4:.0f} Hz added, mel grows by ~{(f_mel[4] - f_mel[3])*4/2:.0f}")
print(f"\n  Because mel is a logarithmic scale, the same rise in frequency produces a bigger mel rise at low frequencies.")
print(f"  So on the mel heatmap the sweep climbs steeply at first and then levels off (the mel axis itself compresses the high frequencies).")
print(f"\n[What it means for human hearing]")
print(f"  • 'many high-frequency rows' on the linear heatmap shows: the Hz axis gives high frequencies lots of resolution the ear doesn't actually care about")
print(f"  • the 'steep-then-flat' curve on the mel heatmap shows: the information has been reorganized along the ear's perceptual dimension, keeping rich low-frequency detail")

## 3. Parameter experiment: linear spectrogram vs mel spectrogram of a chirp

### Why decibels (dB)? And why the factors differ

Spectrum values often span a huge range (from 1e-10 to 1e3); plot them on a linear scale and the small values get drowned out. **The decibel (dB)** is a logarithmic scale that "compresses" that enormous range into an easy-to-read range of about 0–100.

**Key formulas**:
- **magnitude spectrum → dB**: `S_db = 20 * log10(S + ε)`
- **power spectrum → dB**: `M_db = 10 * log10(M + ε)`

Why are the factors different? Because power = magnitude squared. The derivation:

```
if M = S² (power = magnitude²)
then log10(M) = log10(S²) = 2·log10(S)
so 10·log10(M) = 10·2·log10(S) = 20·log10(S)
```

Conclusion: **10·log10(magnitude²) = 20·log10(magnitude)** — the two give exactly the same number; the different factor just accounts for the square. Whichever formula you use, the perceived result is the same. In practice we agree that:
- if the input is a **magnitude spectrum** → use 20·log10
- if the input is a **power spectrum** → use 10·log10

### Dynamic range and vmin: why subtract 60 dB?

What does `vmin = S_db.max() - 60` mean in the heatmap?

**Another name for dB is "relative ratio."** 60 dB below the peak means 1/1000 of the amplitude ($10^{-60/20} = 1/1000$), i.e. one millionth of the power. This number is a **psychoacoustic convention**:
- the dynamic range the ear can perceive is about 120 dB (from the quietest sound you can hear to the loudest before it becomes painful)
- but in real speech or music spectra, the meaningful part usually lies within a 40–60 dB range
- anything more than 60 dB down is mostly **noise** (the recording equipment's noise floor, background wind, and so on)

Setting `vmin = max - 60` is meant to **ignore signals that are too weak and make medium-strength signals visible**. With `max - 80` you'd see more noise (a "busier" picture); with `max - 40` you'd lose detail. 60 is an empirically comfortable value.

### The chirp: a linear upward frequency sweep

A "chirp" is a **frequency-sweep signal** — its frequency rises linearly over time, for example from 200 Hz to 8000 Hz over 2 seconds. Signals like this are especially good for testing, because:
1. their frequency content changes monotonically over time (easy to track)
2. the energy is equal at every frequency (handy for comparing how the Hz axis and the mel axis compress things)

In [ ]:
# ── Standalone mathematical assertions: checking the core properties of time–frequency analysis (no student code needed) ──
import numpy as np

# 1. STFT frame-count formula (aurora's stft defaults to center=True: reflection-pad n_fft//2 at both ends)
N_sig = len(sig)
pad = n_fft // 2
n_frames_calc = 1 + (N_sig + 2 * pad - n_fft) // hop_length  # with even n_fft this = 1 + N//hop
assert S.shape[0] == n_frames_calc, f"frame count should be {n_frames_calc}, got {S.shape[0]}"
print(f"1 ✅  frame-count formula (center=True): 1+(N+2·(n_fft//2)-n_fft)//hop = 1+{N_sig}//{hop_length} = {n_frames_calc}")

# 2. STFT frequency dimension: n_fft//2+1 (one-sided spectrum)
n_bins = n_fft // 2 + 1
assert S.shape[1] == n_bins, f"frequency bins should be {n_bins}, got {S.shape[1]}"
print(f"2 ✅  one-sided spectrum bins = n_fft//2+1 = {n_fft}//2+1 = {n_bins}")

# 3. Mel filterbank: shape=(n_mels, n_bins), non-negative weights
fb = mel_filterbank(n_mels=n_mels, n_fft=n_fft, sample_rate=sr, fmin=0.0, fmax=fmax)
assert fb.shape == (n_mels, n_bins), f"fb shape={fb.shape}, expected ({n_mels},{n_bins})"
assert np.all(fb >= 0), "mel filter weights should all be non-negative"
print(f"3 ✅  mel_filterbank shape={fb.shape}, all weights non-negative")

# 4. The mel spectrogram's shape matches n_mels
assert M.shape == (S.shape[0], n_mels), f"mel spectrogram shape={M.shape}, expected ({S.shape[0]},{n_mels})"
assert np.all(M >= 0), "the power spectrum (mel energy) should be all non-negative"
print(f"4 ✅  mel spectrogram shape={M.shape} (n_frames={S.shape[0]}, n_mels={n_mels}), values non-negative")

# 5. log-mel has no nan/inf (the ε guard works)
log_mel = np.log(M + 1e-8)
assert np.all(np.isfinite(log_mel)), f"log-mel contains nan/inf! min={log_mel.min():.2f}"
print(f"5 ✅  log-mel all finite: min={log_mel.min():.2f}, max={log_mel.max():.2f} (natural log)")

# 6. mel energy = magnitude² @ fb.T (checking the direction of the matrix product)
power_check = S[:5] ** 2  # (5, n_bins)
mel_check = power_check @ fb.T   # (5, n_mels)
assert mel_check.shape == (5, n_mels)
print(f"6 ✅  power({5},{n_bins}) @ fb.T({n_bins},{n_mels}) = mel_energy({5},{n_mels})")


## How does a mel filter compute energy by matrix multiplication? A detailed example

Earlier we said "power spectrum @ mel filter matrix = mel energy." What does that actually mean?

### A tiny example: 3 frequency bins × 2 mel filters

Suppose:
- **spectrum** (magnitude): one frame's spectrum has 3 frequency bins with values `[10, 20, 15]` (say, at 100 Hz, 1000 Hz and 8000 Hz)
- **power spectrum**: squaring gives `[100, 400, 225]`
- **mel filterbank** (shape: 2 filters × 3 bins): set up as the matrix

```
fb = [
  [0.5,  0.5,  0.0],   # mel filter 1 (low frequencies)
  [0.0,  0.3,  0.7]    # mel filter 2 (mid-high frequencies)
]
```

**Working it out by hand**:

In [ ]:
# Tiny example: how matrix multiplication computes mel energy
print("[Example: 3 frequency bins × 2 mel filters]\n")

# Magnitude and power of one frame's spectrum
S_frame = np.array([10, 20, 15])  # magnitude spectrum
P_frame = S_frame ** 2  # power spectrum: [100, 400, 225]

print(f"1. Power spectrum (1 frame): {P_frame}")

# Mel filter matrix (illustrative)
fb_demo = np.array([
    [0.5,  0.5,  0.0],  # filter 1: responds on the first two bins (low frequencies)
    [0.0,  0.3,  0.7]   # filter 2: responds on the last two bins (high frequencies)
])

print(f"\n2. Mel filter matrix (2 filters × 3 bins):")
print(fb_demo)

# Method 1: one at a time (weighted sum)
mel_energy_0 = P_frame[0] * fb_demo[0, 0] + P_frame[1] * fb_demo[0, 1] + P_frame[2] * fb_demo[0, 2]
mel_energy_1 = P_frame[0] * fb_demo[1, 0] + P_frame[1] * fb_demo[1, 1] + P_frame[2] * fb_demo[1, 2]

print(f"\n3. Weighted sums (one filter at a time):")
print(f"   mel filter 1: {P_frame[0]}×{fb_demo[0,0]} + {P_frame[1]}×{fb_demo[0,1]} + {P_frame[2]}×{fb_demo[0,2]}")
print(f"              = {P_frame[0]*fb_demo[0,0]:.0f} + {P_frame[1]*fb_demo[0,1]:.0f} + {P_frame[2]*fb_demo[0,2]:.0f}")
print(f"              = {mel_energy_0:.0f}")
print(f"\n   mel filter 2: {P_frame[0]}×{fb_demo[1,0]} + {P_frame[1]}×{fb_demo[1,1]} + {P_frame[2]}×{fb_demo[1,2]}")
print(f"              = {P_frame[0]*fb_demo[1,0]:.0f} + {P_frame[1]*fb_demo[1,1]:.0f} + {P_frame[2]*fb_demo[1,2]:.0f}")
print(f"              = {mel_energy_1:.0f}")

# Method 2: matrix multiplication
mel_energy_matrix = P_frame @ fb_demo.T  # (3,) @ (2, 3).T = (3,) @ (3, 2) = (2,)

print(f"\n4. Result of the matrix product:")
print(f"   power_frame @ fb.T = {P_frame} @ ")
print(f"                        [{fb_demo[0,0]}, {fb_demo[1,0]}")
print(f"                         {fb_demo[0,1]}, {fb_demo[1,1]}")
print(f"                         {fb_demo[0,2]}, {fb_demo[1,2]}]")
print(f"                      = {mel_energy_matrix}")

print(f"\n✓ Confirmed: one at a time [{mel_energy_0:.0f}, {mel_energy_1:.0f}] == matrix product {mel_energy_matrix}")

print(f"\n[Making sense of it]")
print(f"  • each mel filter is a row holding that filter's weights for the 3 frequency bins")
print(f"  • multiplying the power spectrum (3,) by the transposed fb (3, 2) amounts to:")
print(f"    - multiplying each element of the power spectrum by the matching weight in a column of fb.T")
print(f"    - summing each column → 1 mel energy")
print(f"  • the result is a (2,) vector: this frame's energy in the 2 mel channels")
print(f"\n[Generalizing]")
print(f"  with n_frames frames, shape (n_frames, 3), times fb.T of shape (3, 2)")
print(f"  the result is (n_frames, 2), each row one frame's mel energies")

## Appendix · Why does the next lesson do a DCT? (a preview of L49)

### The problem of correlated mel channels

L48 has already cut sound into a frame-by-frame spectrogram and squeezed the linear frequency axis into a mel axis. It looks perfect, but there's a hidden problem:

**Mel channels are highly correlated with each other.** Neighboring mel channels overlap on the frequency axis, so they often "repeat" the same thing, with correlation coefficients often close to 0.9. That's wasteful — if you have 80 mel channels, the truly independent information may be far less than 80.

Let's check.

In [ ]:
# Demo: correlation between mel channels
print("[Mel channel correlation analysis]\n")

# Use the mel spectrogram M generated from the chirp above (shape: n_frames × n_mels)
# Compute the correlation-coefficient matrix of the mel spectrogram
mel_corr = np.corrcoef(M.T)  # (n_mels, n_mels)

print(f"mel spectrogram matrix shape: {M.shape}  (n_frames={M.shape[0]}, n_mels={M.shape[1]})")
print(f"correlation matrix shape: {mel_corr.shape}")

# Find the correlation between neighboring channels
neighbor_corr = []
for m in range(n_mels - 1):
    r = mel_corr[m, m + 1]  # correlation coefficient between channel m and channel m+1
    neighbor_corr.append(r)

print(f"\nCorrelation coefficients between neighboring mel channels (first 20):")
print(f"  {neighbor_corr[:20]}")
print(f"\nStatistics:")
print(f"  • mean correlation: {np.mean(neighbor_corr):.3f}")
print(f"  • min correlation:  {np.min(neighbor_corr):.3f}")
print(f"  • max correlation:  {np.max(neighbor_corr):.3f}")

# Plot the correlation matrix as a heatmap
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: the full correlation matrix
ax = axes[0]
im = ax.imshow(mel_corr, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
ax.set_xlabel('Mel channel j')
ax.set_ylabel('Mel channel i')
ax.set_title(f'Full correlation matrix of the mel channels\n(diagonal is 1, neighbors close to 0.9)')
plt.colorbar(im, ax=ax, label='correlation r')

# Right: how the neighbor correlation varies
ax = axes[1]
ax.plot(neighbor_corr, 'b-', lw=2, marker='o', markersize=3)
ax.axhline(np.mean(neighbor_corr), color='r', ls='--', lw=2, label=f'mean = {np.mean(neighbor_corr):.3f}')
ax.set_xlabel('Channel pair (m, m+1)')
ax.set_ylabel('Correlation r')
ax.set_title('Correlation between neighboring mel channels')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print(f"\n[Explaining what we see]")
print(f"  Why is the correlation between neighboring channels so high (~0.9)?")
print(f"  • each mel filter is a triangle that overlaps heavily with its neighbors")
print(f"  • so neighboring channels capture very similar frequency content")
print(f"  • result: the channels are redundant, and the independent information in 80 channels << 80")
print(f"\n[What the DCT does]")
print(f"  The discrete cosine transform (DCT) is a rotation of coordinates that takes the highly correlated mel energies")
print(f"  into a new coordinate system where the new coordinates (the MFCC coefficients) are nearly uncorrelated.")
print(f"  • the low-order MFCC coefficients (1–13) hold ~95% of the information")
print(f"  • the high-order coefficients are often noise")
print(f"  That's why speech recognition often uses 13-dimensional MFCCs rather than 80-dimensional mel energies.")

## Wrap-up

`stft` + `magnitude_spectrogram` first give a linear magnitude spectrogram of shape `(n_frames, n_fft//2+1)`; `mel_filterbank` then compresses it into a mel spectrogram of shape `(n_frames, n_mels)`. Both come from the `aurora.audio.stft` and `aurora.audio.mel` modules. The mel axis is perceptually more even, so on the same amount of screen the low-frequency detail is kept while the high frequencies are deliberately merged. Next (**L49**) we run a DCT on top of the log-mel to get MFCCs, the cepstral coefficients.

---

→ **Next lesson** · [L49 · The DCT-II](L49_dct.ipynb)

> Next lesson covers **the DCT-II**: why it decorrelates, and a pure-NumPy replacement for `scipy.fft.dct`.